## Summary & Key Insights

### What We Learned

1. **Seasonal Vegetation Dynamics**: The summer NDVI is significantly higher than winter, reflecting the Mediterranean climate where most vegetation responds strongly to warm temperatures and seasonal moisture availability.

2. **2026 as a Wet Year**: With above-average winter precipitation, vegetation in responsive areas (agricultural land, grasslands) shows enhanced photosynthetic activity in summer. Areas with lower seasonal variation may indicate evergreen forests or chronic water stress.

3. **Land Use Patterns**: 
   - **Dense Veg.**: Mostly riparian zones and healthy forests
   - **Healthy Veg.**: Agricultural areas and managed forests
   - **Moderate Veg.**: Shrublands and sparse grazing lands
   - **Sparse/Urban**: Built-up areas and bare soils
   - **Water/Barren**: Coastal areas and rocky outcrops

4. **Climate Resilience**: This baseline from a wet year provides a reference point for comparing future droughts or wet years.

### Next Steps for Your Portfolio

- **Interactive Map**: Deploy the GeoPackage to an interactive web map (Folium, Leaflet.js)
- **Streamlit Dashboard**: Create a real-time tool for exploring NDVI by municipality
- **Publication**: Write a blog post explaining findings with the visualizations
- **Comparative Analysis**: Repeat this analysis for other Mediterranean islands or time periods

In [ ]:
gpkg_path = os.path.join(output_dir, 'ndvi_mallorca.gpkg')

if gdf_summer is not None:
    gdf_summer.to_file(gpkg_path, layer='ndvi_summer_classified', 
                       driver='GPKG')
    print(f"✓ Layer: ndvi_summer_classified  ({len(gdf_summer):,} polygons)")

if gdf_winter is not None:
    gdf_winter.to_file(gpkg_path, layer='ndvi_winter_classified', 
                       driver='GPKG')
    print(f"✓ Layer: ndvi_winter_classified  ({len(gdf_winter):,} polygons)")

municipis.to_file(gpkg_path, layer='municipis_boundary', driver='GPKG')
print("✓ Layer: municipis_boundary")

print(f"\n{'='*60}")
print(f"GeoPackage saved: {gpkg_path}")
print(f"{'='*60}")

In [ ]:
def vectorize_ndvi_classes(ndvi_arr, transform, crs):
    """Convert classified NDVI raster to polygons (GeoDataFrame)"""
    classified = np.zeros(ndvi_arr.shape, dtype=np.uint8)
    for i in range(len(CLASS_BINS) - 1):
        mask = (
            (~np.isnan(ndvi_arr)) & 
            (ndvi_arr >= CLASS_BINS[i]) & 
            (ndvi_arr < CLASS_BINS[i + 1])
        )
        classified[mask] = i + 1  # 1-based (0 = nodata)
    
    records = []
    for geom, val in shapes(classified, mask=(classified > 0), 
                            transform=transform):
        idx = int(val) - 1
        if 0 <= idx < len(CLASS_LABELS):
            records.append({
                'geometry': shapely_shape(geom),
                'class_id': int(val),
                'class_label': CLASS_LABELS[idx],
            })
    
    if not records:
        return None
    gdf = gpd.GeoDataFrame(records, crs=crs)
    return gdf[gdf.geometry.is_valid].copy()

print("Vectorizing classified NDVI...")
gdf_summer = vectorize_ndvi_classes(ndvi_summer, t_summer, crs)
gdf_winter = vectorize_ndvi_classes(ndvi_winter, t_winter, crs)

print(f"Summer: {len(gdf_summer):,} polygons")
print(f"Winter: {len(gdf_winter):,} polygons")

## 13. Export Results: GeoPackage

Convert classified rasters to vector polygons and save with municipality boundaries in GeoPackage format.

In [ ]:
def save_raster(array, transform, crs, path, nodata=-9999.0):
    """Write float32 array to compressed GeoTIFF"""
    profile = {
        'driver': 'GTiff', 'dtype': 'float32', 'count': 1,
        'height': array.shape[0], 'width': array.shape[1],
        'crs': crs, 'transform': transform,
        'nodata': nodata, 'compress': 'lzw',
    }
    out = np.where(np.isnan(array), nodata, array).astype('float32')
    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(out, 1)
    print(f"✓ Saved: {os.path.basename(path)}")

# Export rasters
output_dir = os.path.join(os.getcwd(), 'output')
os.makedirs(output_dir, exist_ok=True)

save_raster(ndvi_summer, t_summer, crs, 
            os.path.join(output_dir, 'ndvi_summer.tif'))
save_raster(ndvi_winter, t_winter, crs, 
            os.path.join(output_dir, 'ndvi_winter.tif'))
save_raster(ndvi_diff, t_summer, crs, 
            os.path.join(output_dir, 'ndvi_difference.tif'))

print("\nAll GeoTIFF rasters exported successfully!")

## 12. Export Results: GeoTIFF Rasters

Save continuous NDVI and difference maps as compressed GeoTIFF files for external GIS tools.

In [ ]:
cmap_cls = mcolors.ListedColormap(CLASS_COLORS)
norm_cls = mcolors.BoundaryNorm(range(len(CLASS_LABELS) + 1), cmap_cls.N)
legend_patches = [
    Patch(facecolor=c, label=l) 
    for c, l in zip(CLASS_COLORS, CLASS_LABELS)
]

fig2, axes2 = plt.subplots(1, 2, figsize=(16, 8))
fig2.suptitle('NDVI Classification — Mallorca', fontsize=16, fontweight='bold')

for ax, data, title in zip(
    axes2,
    [cls_summer[:rows, :cols], cls_winter[:rows, :cols]],
    [f'Summer ({summer_date})', f'Winter ({winter_date})'],
):
    ax.imshow(data, cmap=cmap_cls, norm=norm_cls, interpolation='nearest')
    ax.set_title(title, fontsize=13)
    ax.axis('off')
    ax.legend(handles=legend_patches, loc='lower right', fontsize=9,
              framealpha=0.9, title='NDVI Class')

plt.tight_layout()
plt.show()

## 11. Visualize Results: Classified Maps

Display vegetation categories using the predefined color scheme.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 8))
fig.suptitle('NDVI Analysis — Mallorca | Landsat 8 C2 L2', 
             fontsize=16, fontweight='bold')

plot_specs = [
    (ndvi_summer[:rows, :cols], f'Summer NDVI\n({summer_date})', 
     plt.cm.RdYlGn, -0.2, 0.8, 'NDVI'),
    (ndvi_winter[:rows, :cols], f'Winter NDVI\n({winter_date})', 
     plt.cm.RdYlGn, -0.2, 0.8, 'NDVI'),
    (ndvi_diff, 'Difference\n(Summer − Winter)', 
     plt.cm.RdBu, -0.4, 0.4, 'ΔNDVI'),
]

for ax, (data, title, cmap, vmin, vmax, cbar_label) in zip(axes, plot_specs):
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, 
                   interpolation='bilinear')
    ax.set_title(title, fontsize=13, pad=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, 
                 label=cbar_label, shrink=0.85)

plt.tight_layout()
plt.show()

## 10. Visualize Results: Continuous NDVI Maps

Create a multi-panel figure showing summer NDVI, winter NDVI, and their difference.

In [ ]:
def classify_ndvi(ndvi_arr):
    """Classify NDVI into vegetation categories"""
    classified = np.full(ndvi_arr.shape, np.nan)
    for i in range(len(CLASS_BINS) - 1):
        mask = (
            (~np.isnan(ndvi_arr)) & 
            (ndvi_arr >= CLASS_BINS[i]) & 
            (ndvi_arr < CLASS_BINS[i + 1])
        )
        classified[mask] = i
    return classified

cls_summer = classify_ndvi(ndvi_summer)
cls_winter = classify_ndvi(ndvi_winter)

# Count pixels in each class
print("Summer Classification:")
for i, label in enumerate(CLASS_LABELS):
    count = int(np.sum(cls_summer == i))
    pct = 100 * count / np.sum(cls_summer >= 0)
    print(f"  {label:20s}: {count:8,} px ({pct:5.1f}%)")

print("\nWinter Classification:")
for i, label in enumerate(CLASS_LABELS):
    count = int(np.sum(cls_winter == i))
    pct = 100 * count / np.sum(cls_winter >= 0)
    print(f"  {label:20s}: {count:8,} px ({pct:5.1f}%)")

## 9. Classify NDVI into Vegetation Categories

Bin NDVI values into five discrete classes using predefined thresholds.

In [ ]:
# Align dimensions (may differ slightly after mosaicking)
rows = min(ndvi_summer.shape[0], ndvi_winter.shape[0])
cols = min(ndvi_summer.shape[1], ndvi_winter.shape[1])

ndvi_diff = ndvi_summer[:rows, :cols] - ndvi_winter[:rows, :cols]

print("NDVI Difference (Summer − Winter):")
print(f"  Mean: {np.nanmean(ndvi_diff):.3f}")
print(f"  Std Dev: {np.nanstd(ndvi_diff):.3f}")
print(f"  Pixels greener in summer: {int(np.nansum(ndvi_diff > 0)):,}")
print(f"  Pixels greener in winter: {int(np.nansum(ndvi_diff < 0)):,}")
print(f"\nInterpretation (2026: a wet year):")
print(f"  High positive values: Strong summer greening response to water availability")
print(f"  Values near zero: Year-round stable vegetation or evergreen forests")
print(f"  Negative values: Winter growth (rare in Mediterranean, indicates irrigation)")

## 8. Compute Seasonal Difference

Calculate Summer NDVI − Winter NDVI to highlight areas with strong seasonal vegetation dynamics.

In [ ]:
def compute_ndvi(red, nir):
    """Calculate NDVI = (NIR - Red) / (NIR + Red)"""
    invalid = (red <= 0) | (nir <= 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ndvi = np.where(invalid, np.nan, (nir - red) / (nir + red))
    ndvi = np.clip(ndvi, -1.0, 1.0)
    return ndvi

# Compute NDVI
ndvi_summer = compute_ndvi(red_summer, nir_summer)
ndvi_winter = compute_ndvi(red_winter, nir_winter)

print("Summer NDVI Statistics:")
print(f"  Mean: {np.nanmean(ndvi_summer):.3f}")
print(f"  Std Dev: {np.nanstd(ndvi_summer):.3f}")
print(f"  Min: {np.nanmin(ndvi_summer):.3f}")
print(f"  Max: {np.nanmax(ndvi_summer):.3f}")
print(f"  Valid pixels: {int(np.sum(~np.isnan(ndvi_summer))):,}")

print("\nWinter NDVI Statistics:")
print(f"  Mean: {np.nanmean(ndvi_winter):.3f}")
print(f"  Std Dev: {np.nanstd(ndvi_winter):.3f}")
print(f"  Min: {np.nanmin(ndvi_winter):.3f}")
print(f"  Max: {np.nanmax(ndvi_winter):.3f}")
print(f"  Valid pixels: {int(np.sum(~np.isnan(ndvi_winter))):,}")

## 7. Calculate NDVI

The Normalized Difference Vegetation Index is calculated as: **NDVI = (NIR − Red) / (NIR + Red)**

Valid NDVI values range from −1 to +1, where positive values indicate vegetation.

In [ ]:
def mosaic_and_clip_band(tif_paths, clip_geojson):
    """Merge multiple TIFs into one mosaic and clip to island outline"""
    datasets = [rasterio.open(p) for p in tif_paths]
    try:
        mosaic_arr, mosaic_transform = merge(datasets)
        src_crs = datasets[0].crs
        src_dtype = datasets[0].dtypes[0]
        
        # rasterio.mask requires a file-like source
        profile = datasets[0].profile.copy()
        profile.update(
            height=mosaic_arr.shape[1],
            width=mosaic_arr.shape[2],
            transform=mosaic_transform,
            count=1,
        )
        with MemoryFile() as memfile:
            with memfile.open(**profile) as mem:
                mem.write(mosaic_arr)
                clipped, clipped_transform = rasterio.mask.mask(
                    mem, clip_geojson, crop=True, nodata=0
                )
    finally:
        for ds in datasets:
            ds.close()
    
    band = clipped[0].astype(np.float32)
    # Apply Landsat C2 L2 scale factor
    if src_dtype == 'uint16':
        band = band * L2_SCALE + L2_OFFSET
    
    return band, clipped_transform, src_crs

print("Mosaicking summer bands...")
red_summer, t_summer, crs = mosaic_and_clip_band(summer_b4, island_geojson)
nir_summer, _, _ = mosaic_and_clip_band(summer_b5, island_geojson)
print(f"  Shape: {red_summer.shape}")

print("Mosaicking winter bands...")
red_winter, t_winter, _ = mosaic_and_clip_band(winter_b4, island_geojson)
nir_winter, _, _ = mosaic_and_clip_band(winter_b5, island_geojson)
print(f"  Shape: {red_winter.shape}")

## 6. Mosaic, Clip, and Scale Bands

For each season, we'll merge multiple tiles into a single mosaic, clip to the island, and apply the Landsat C2 L2 scale factor.

In [ ]:
def extract_acq_date(filepath):
    """Extract YYYYMMDD from Landsat filename (field 3)"""
    parts = os.path.basename(filepath).split('_')
    return parts[3] if len(parts) > 3 else ''

# Categorize by season
summer_b4, summer_b5 = [], []
winter_b4, winter_b5 = [], []
summer_date, winter_date = None, None

for path in b4_files:
    date_str = extract_acq_date(path)
    try:
        month = int(date_str[4:6])
        is_summer = 5 <= month <= 10
        
        b5_path = path.replace('_SR_B4', '_SR_B5')
        
        if is_summer:
            summer_b4.append(path)
            summer_b5.append(b5_path)
            if summer_date is None:
                summer_date = date_str
        else:
            winter_b4.append(path)
            winter_b5.append(b5_path)
            if winter_date is None:
                winter_date = date_str
    except ValueError:
        pass

print(f"Summer scenes: {len(summer_b4)} ({summer_date})")
print(f"Winter scenes: {len(winter_b4)} ({winter_date})")

## 5. Detect and Split Scenes by Season

Extract acquisition dates from filenames to categorize scenes as summer (May-Oct) or winter (Nov-Apr).

In [ ]:
# Get raster CRS
with rasterio.open(b4_files[0]) as src:
    raster_crs = src.crs

# Reproject municipis if needed
if municipis.crs is None:
    municipis = municipis.set_crs("EPSG:4326")
    print("Set municipis CRS to EPSG:4326 (GeoJSON default)")

if municipis.crs != raster_crs:
    municipis = municipis.to_crs(raster_crs)
    print(f"Reprojected municipis to {raster_crs}")
else:
    print(f"CRS already aligned: {raster_crs}")

# Create island outline by merging all municipalities
island_geom = unary_union(municipis.geometry)
print(f"\nIsland outline created ({island_geom.geom_type})")
print(f"Area: {island_geom.area:,.0f} sq units")

island_geojson = [island_geom.__geo_interface__]

## 4. Align CRS and Create Island Mask

We'll reproject the municipality boundaries to match the raster CRS and merge them into a single island outline.

In [ ]:
municipis_json = os.path.join(DATA_RAW, 'municipis.json')
municipis = gpd.read_file(municipis_json)

print(f"Loaded {len(municipis)} municipalities")
print(f"CRS: {municipis.crs}")
print(f"\nGeometry types:")
print(municipis.geometry.geom_type.value_counts())
print(f"\nBounds: {municipis.total_bounds}")

# Display first few municipalities
print(f"\nFirst 3 municipalities:")
print(municipis[['name', 'geometry']].head(3))

## 3. Load Administrative Boundaries

We'll load Mallorca's municipality boundaries as GeoJSON to define our study area.

In [ ]:
# Find all SR_B4 files (Red band) for both summer and winter
b4_files = sorted(
    glob.glob(os.path.join(DATA_RAW, '*_SR_B4.TIF')) +
    glob.glob(os.path.join(DATA_RAW, '*_SR_B4.tif'))
)

print(f"Found {len(b4_files)} Red (B4) band files:")
for f in b4_files[:3]:
    print(f"  - {os.path.basename(f)}")
if len(b4_files) > 3:
    print(f"  ... and {len(b4_files) - 3} more")

# Examine the first band file
with rasterio.open(b4_files[0]) as src:
    print(f"\nBand metadata:")
    print(f"  CRS: {src.crs}")
    print(f"  Resolution: {src.res}")
    print(f"  Bounds: {src.bounds}")
    print(f"  Data type: {src.dtypes[0]}")
    print(f"  Size: {src.width} x {src.height} pixels")

## 2. Load and Explore Landsat Data

Let's load the Landsat 8 Collection 2 Level-2 surface reflectance bands. The Red (B4) and Near-Infrared (B5) bands are essential for computing NDVI.

In [ ]:
# Landsat C2 L2 scale factors for converting uint16 to surface reflectance
L2_SCALE = 0.0000275
L2_OFFSET = -0.2

# NDVI classification thresholds and labels
CLASS_BINS = [-1.0, 0.0, 0.2, 0.4, 0.6, 1.01]
CLASS_LABELS = ['Water/Barren', 'Sparse/Urban', 'Moderate Veg.', 
                'Healthy Veg.', 'Dense Veg.']
CLASS_COLORS = ['#4575b4', '#ffffbf', '#a6d96a', '#66bd63', '#1a7837']

print("Landsat Collection 2 L2 scale factors loaded")
print("NDVI classification schema defined")

## 1. Import Required Libraries

We'll use specialized geospatial libraries to handle satellite imagery and vector data.

In [ ]:
import os
import glob
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import rasterio
import rasterio.mask
from rasterio.features import shapes
from rasterio.merge import merge
from rasterio.io import MemoryFile
import geopandas as gpd
from shapely.geometry import shape as shapely_shape
from shapely.ops import unary_union, polygonize

# Suppress noisy warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Set up paths
BASE_DIR = os.path.dirname(os.path.abspath('../../'))
DATA_RAW = os.path.join(BASE_DIR, 'data', 'raw')
DATA_OUT = os.path.join(BASE_DIR, 'data', 'output')

print(f"Working directory: {DATA_RAW}")
print(f"Output directory: {DATA_OUT}")

# NDVI Analysis — Mallorca (Summer vs Winter)

## Comprehensive Walkthrough

This notebook demonstrates the complete workflow for analyzing vegetation health on Mallorca using Landsat 8 satellite data. We'll compare NDVI (Normalized Difference Vegetation Index) between summer and winter to understand seasonal vegetation dynamics in a notably wet year (2026).